# Week 5 — Medallion Architecture Notebook (Elite)

Simulate Bronze → Silver → Gold pipeline using insurance data.

Goal: Understand how the SAME data evolves across layers.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

con = duckdb.connect()

## Load Raw Data (Bronze Layer)

In [ ]:
df = pd.read_csv("insurance.csv")

# Simulate messy data
df.loc[0, "region"] = "NE"
df.loc[1, "region"] = "north-east"
df.loc[2, "charges"] = None

bronze = df.copy()
bronze.head()

### Bronze Characteristics
- raw
- messy
- unclean
- preserved

In [ ]:
bronze["region"].value_counts(dropna=False)

## Silver Layer — Cleaning

In [ ]:
silver = bronze.copy()

# Clean region
region_map = {
    "NE": "northeast",
    "north-east": "northeast"
}
silver["region"] = silver["region"].replace(region_map).str.lower()

# Remove null charges
silver = silver[silver["charges"].notna()]

# Derived column
def age_group(age):
    if age < 30:
        return "young"
    elif age < 60:
        return "adult"
    else:
        return "senior"

silver["age_group"] = silver["age"].apply(age_group)

silver.head()

In [ ]:
silver["region"].value_counts()

### Insight
Cleaning merges categories → affects analytics

## Compare Bronze vs Silver

In [ ]:
compare = pd.DataFrame({
    "bronze_regions": bronze["region"].value_counts(),
    "silver_regions": silver["region"].value_counts()
}).fillna(0)

compare

## Gold Layer — Star Schema

In [ ]:
# dim_region
dim_region = silver[["region"]].drop_duplicates().reset_index(drop=True)
dim_region["region_id"] = range(1, len(dim_region)+1)

# dim_person
dim_person = silver[["age","gender","bmi","children","age_group"]].copy()
dim_person["person_id"] = range(1, len(dim_person)+1)

# fact
fact = silver.copy().reset_index(drop=True)
fact["person_id"] = dim_person["person_id"]

fact = fact.merge(dim_region, on="region")

fact = fact[["person_id","region_id","charges"]]

fact.head()

### Insight
Gold = structured, optimized for queries

## Register Tables

In [ ]:
con.register("fact", fact)
con.register("dim_region", dim_region)
con.register("dim_person", dim_person)

## OLAP Query — Revenue by Region

In [ ]:
q = con.execute(
"""
SELECT r.region, AVG(f.charges) avg_charge
FROM fact f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region
"""
).df()

q

In [ ]:
plt.figure()
plt.bar(q["region"], q["avg_charge"])
plt.title("Avg Charges by Region (Gold Layer)")
plt.show()

## Key Comparison


| Layer | Purpose |
|------|--------|
| Bronze | raw data |
| Silver | cleaned data |
| Gold | analytical model |


## Final Insight


Same data → different value at each layer:

- Bronze → traceability  
- Silver → trust  
- Gold → decisions  

👉 Medallion = structured evolution of data
